Java+Install

In [13]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

!pip install pyspark==3.5.1 delta-spark==3.1.0 azure-storage-blob -q
print("✅ Ready")

✅ Ready


Spark Session

In [14]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = SparkSession.builder \
    .appName("AzureETLPipeline") \
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("✅ Spark version:", spark.version)

✅ Spark version: 3.5.1


Download files from ADLS into Colab using Azure SDK

In [17]:
from azure.storage.blob import BlobServiceClient

STORAGE_ACCOUNT = "learningetl"
ACCESS_KEY = "YOUR ACCESS KEY HERE"
CONTAINER = "medallion"

client = BlobServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.blob.core.windows.net",
    credential=ACCESS_KEY
)

for f in ["customers", "products", "transactions", "stores"]:
    blob = client.get_blob_client(container=CONTAINER, blob=f"raw/{f}.json")
    with open(f"/tmp/{f}.json", "wb") as fp:
        fp.write(blob.download_blob().readall())
    print(f"✅ Downloaded {f}.json")

✅ Downloaded customers.json
✅ Downloaded products.json
✅ Downloaded transactions.json
✅ Downloaded stores.json


Bronze Layer:Read from local files

In [18]:
df_customers    = spark.read.option("multiline", "true").json("/tmp/customers.json")
df_products     = spark.read.option("multiline", "true").json("/tmp/products.json")
df_transactions = spark.read.option("multiline", "true").json("/tmp/transactions.json")
df_stores       = spark.read.option("multiline", "true").json("/tmp/stores.json")

print("✅ Bronze layer loaded successfully")
print(f"   Customers    : {df_customers.count()} rows")
print(f"   Products     : {df_products.count()} rows")
print(f"   Transactions : {df_transactions.count()} rows")
print(f"   Stores       : {df_stores.count()} rows")

df_customers.printSchema()
df_transactions.show(5, truncate=False)

✅ Bronze layer loaded successfully
   Customers    : 15 rows
   Products     : 15 rows
   Transactions : 25 rows
   Stores       : 4 rows
root
 |-- age: long (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- email: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- membership: string (nullable = true)
 |-- name: string (nullable = true)
 |-- signup_date: string (nullable = true)

+-----------+--------+--------------+----------+--------+--------+----------------+--------------+
|customer_id|discount|payment_method|product_id|quantity|store_id|transaction_date|transaction_id|
+-----------+--------+--------------+----------+--------+--------+----------------+--------------+
|1          |10.0    |Credit Card   |101       |2       |1       |2024-01-05      |1001          |
|3          |0.0     |Debit Card    |103       |1       |2       |2024-01-06      |1002          |
|2          |5.0

Cell 5:Silver Layer:Clean & Transform

In [19]:
from pyspark.sql.functions import col, to_date, upper, trim
from pyspark.sql.types import IntegerType, DoubleType

df_silver_customers = df_customers \
    .fillna({"age": 0, "email": "unknown@email.com"}) \
    .withColumn("customer_id",  col("customer_id").cast(IntegerType())) \
    .withColumn("age",          col("age").cast(IntegerType())) \
    .withColumn("name",         trim(col("name"))) \
    .withColumn("membership",   upper(trim(col("membership")))) \
    .withColumn("signup_date",  to_date(col("signup_date"), "yyyy-MM-dd")) \
    .dropDuplicates(["customer_id"])

df_silver_products = df_products \
    .fillna({"stock_quantity": 0, "rating": 0.0}) \
    .withColumn("product_id",     col("product_id").cast(IntegerType())) \
    .withColumn("price",          col("price").cast(DoubleType())) \
    .withColumn("stock_quantity", col("stock_quantity").cast(IntegerType())) \
    .withColumn("rating",         col("rating").cast(DoubleType())) \
    .withColumn("launch_date",    to_date(col("launch_date"), "yyyy-MM-dd")) \
    .dropDuplicates(["product_id"])

df_silver_transactions = df_transactions \
    .fillna({"discount": 0.0}) \
    .withColumn("transaction_id",   col("transaction_id").cast(IntegerType())) \
    .withColumn("customer_id",      col("customer_id").cast(IntegerType())) \
    .withColumn("product_id",       col("product_id").cast(IntegerType())) \
    .withColumn("store_id",         col("store_id").cast(IntegerType())) \
    .withColumn("quantity",         col("quantity").cast(IntegerType())) \
    .withColumn("discount",         col("discount").cast(DoubleType())) \
    .withColumn("transaction_date", to_date(col("transaction_date"), "yyyy-MM-dd")) \
    .dropDuplicates(["transaction_id"])

df_silver_stores = df_stores \
    .withColumn("store_id",  col("store_id").cast(IntegerType())) \
    .withColumn("open_date", to_date(col("open_date"), "yyyy-MM-dd")) \
    .withColumn("region",    upper(trim(col("region")))) \
    .dropDuplicates(["store_id"])

print("✅ Silver layer transformations complete")
df_silver_transactions.show(5, truncate=False)

✅ Silver layer transformations complete
+-----------+--------+--------------+----------+--------+--------+----------------+--------------+
|customer_id|discount|payment_method|product_id|quantity|store_id|transaction_date|transaction_id|
+-----------+--------+--------------+----------+--------+--------+----------------+--------------+
|1          |10.0    |Credit Card   |101       |2       |1       |2024-01-05      |1001          |
|3          |0.0     |Debit Card    |103       |1       |2       |2024-01-06      |1002          |
|2          |5.0     |PayPal        |110       |1       |1       |2024-01-07      |1003          |
|5          |0.0     |Credit Card   |102       |2       |3       |2024-01-08      |1004          |
|7          |0.0     |Cash          |107       |3       |2       |2024-01-09      |1005          |
+-----------+--------+--------------+----------+--------+--------+----------------+--------------+
only showing top 5 rows



Gold Layer: Join & aggregate

In [20]:
from pyspark.sql.functions import sum as _sum, avg, count, round as _round, expr

df_joined = df_silver_transactions \
    .join(df_silver_products,  on="product_id",  how="left") \
    .join(df_silver_customers, on="customer_id", how="left") \
    .join(df_silver_stores,    on="store_id",    how="left") \
    .withColumn("revenue",
        _round((col("price") * col("quantity")) - col("discount"), 2))

df_gold_by_category = df_joined.groupBy("category").agg(
    count("transaction_id").alias("total_transactions"),
    _sum("quantity").alias("total_units_sold"),
    _round(_sum("revenue"), 2).alias("total_revenue"),
    _round(avg("revenue"), 2).alias("avg_revenue_per_sale")
).orderBy(col("total_revenue").desc())

df_gold_by_region = df_joined.groupBy("region", "store_name").agg(
    count("transaction_id").alias("total_transactions"),
    _round(_sum("revenue"), 2).alias("total_revenue")
).orderBy(col("total_revenue").desc())

df_gold_clv = df_joined.groupBy("membership").agg(
    count("transaction_id").alias("total_orders"),
    _round(_sum("revenue"), 2).alias("total_revenue"),
    _round(avg("revenue"), 2).alias("avg_order_value")
).orderBy(col("total_revenue").desc())

print("✅ Gold layer complete")
print("\n--- Sales by Category ---")
df_gold_by_category.show(truncate=False)
print("\n--- Sales by Region ---")
df_gold_by_region.show(truncate=False)
print("\n--- Customer Lifetime Value ---")
df_gold_clv.show(truncate=False)

✅ Gold layer complete

--- Sales by Category ---
+---------------+------------------+----------------+-------------+--------------------+
|category       |total_transactions|total_units_sold|total_revenue|avg_revenue_per_sale|
+---------------+------------------+----------------+-------------+--------------------+
|Electronics    |5                 |6               |699.95       |139.99              |
|Home Office    |5                 |8               |407.96       |81.59               |
|Footwear       |2                 |3               |348.0        |174.0               |
|Sports         |5                 |13              |325.96       |65.19               |
|Home Appliances|2                 |2               |273.0        |136.5               |
|Accessories    |3                 |3               |199.0        |66.33               |
|Kitchen        |2                 |2               |111.0        |55.5                |
|Health         |1                 |2               |85.0    

 Upload results back to ADLS

In [21]:
import os

# Save to local /tmp first
df_silver_customers.write.format("delta").mode("overwrite").save("/tmp/silver/customers")
df_silver_products.write.format("delta").mode("overwrite").save("/tmp/silver/products")
df_silver_transactions.write.format("delta").mode("overwrite").save("/tmp/silver/transactions")
df_silver_stores.write.format("delta").mode("overwrite").save("/tmp/silver/stores")
df_gold_by_category.write.format("delta").mode("overwrite").save("/tmp/gold/sales_by_category")
df_gold_by_region.write.format("delta").mode("overwrite").save("/tmp/gold/sales_by_region")
df_gold_clv.write.format("delta").mode("overwrite").save("/tmp/gold/customer_lifetime_value")

print("✅ Delta tables saved locally")

# Upload to ADLS
def upload_folder(local_folder, adls_folder):
    for root, dirs, files in os.walk(local_folder):
        for file in files:
            local_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_path, local_folder)
            blob_path = f"{adls_folder}/{relative_path}"
            blob = client.get_blob_client(container=CONTAINER, blob=blob_path)
            with open(local_path, "rb") as f:
                blob.upload_blob(f, overwrite=True)
    print(f"✅ Uploaded {adls_folder}")

upload_folder("/tmp/silver/customers",             "silver/customers")
upload_folder("/tmp/silver/products",              "silver/products")
upload_folder("/tmp/silver/transactions",          "silver/transactions")
upload_folder("/tmp/silver/stores",                "silver/stores")
upload_folder("/tmp/gold/sales_by_category",       "gold/sales_by_category")
upload_folder("/tmp/gold/sales_by_region",         "gold/sales_by_region")
upload_folder("/tmp/gold/customer_lifetime_value", "gold/customer_lifetime_value")

print("🎉 ETL Pipeline Complete! All layers uploaded to ADLS!")

✅ Delta tables saved locally
✅ Uploaded silver/customers
✅ Uploaded silver/products
✅ Uploaded silver/transactions
✅ Uploaded silver/stores
✅ Uploaded gold/sales_by_category
✅ Uploaded gold/sales_by_region
✅ Uploaded gold/customer_lifetime_value
🎉 ETL Pipeline Complete! All layers uploaded to ADLS!
